In [ ]:
# # Inference and Reasoning Pipeline

# This notebook performs disease prediction and generates explainable medical reasoning.

# Tasks:
# - Load trained DenseNet checkpoint
# - Detect thoracic abnormalities
# - Generate reasoning using Phi-3
# - Build inference pipeline for backend API integration

In [5]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
!pip install torch torchvision transformers flask flask-cors pyngrok pillow


In [2]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import io

from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ===== DenseNet Vision Model =====
VISION_WEIGHTS_PATH = "/content/drive/MyDrive/Colab Notebooks/densenet_cxr_checkpoint.pth"

vision_model = models.densenet121(pretrained=False)
vision_model.classifier = nn.Linear(
    vision_model.classifier.in_features, 7
)

checkpoint = torch.load(VISION_WEIGHTS_PATH, map_location=DEVICE)
vision_model.load_state_dict(checkpoint["model_state"])
vision_model = vision_model.to(DEVICE)
vision_model.eval()

print("✅ DenseNet loaded")

LABELS = [
    "lung_opacity",
    "consolidation",
    "pleural_effusion",
    "cardiomegaly",
    "atelectasis",
    "edema",
    "support_devices"
]

image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def extract_findings_pil(img, threshold=0.5):
    img = image_transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = vision_model(img)
        probs = torch.sigmoid(logits).cpu().numpy()[0]

    preds = {k: float(v) for k, v in zip(LABELS, probs)}
    present = [k.replace("_", " ") for k, v in preds.items() if v >= threshold]
    absent  = [k.replace("_", " ") for k, v in preds.items() if v < threshold]

    return preds, present, absent

# ===== Phi-3 Reasoning Model =====
PHI_MODEL = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(PHI_MODEL)
phi_model = AutoModelForCausalLM.from_pretrained(
    PHI_MODEL,
    device_map="auto",
    torch_dtype=torch.float16
)
phi_model.eval()

print("✅ Phi-3 loaded")





Using device: cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


✅ DenseNet loaded


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

✅ Phi-3 loaded


In [9]:
def build_reasoning_prompt(present, absent, question):
    system_prompt = (
        "You are a radiology reasoning assistant.\n"
        "give step by step resoning for the findings in xray in 4 to 5 lines, reason strictly from the findings\n"
        "You MUST output ONLY in the following format:\n\n"
        "Reasoning:\n"
        "provide step-by-step medical reasoning.\n"
        "Conclusion:\n"
        "<one small single sentence>\n\n"

    )

    user_prompt = f"""
Detected findings:
Present: {', '.join(present) if present else 'None'}
Absent: {', '.join(absent) if absent else 'None'}

Question:
{question}
"""

    return f"<|system|>{system_prompt}<|user|>{user_prompt}<|assistant|>"



import re

def reason_with_phi(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(phi_model.device)

    with torch.no_grad():
        output_ids = phi_model.generate(
            **inputs,
            max_new_tokens=350,
            do_sample=False,
            temperature=0.0,
            top_p=1.0
        )

    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Remove assistant tag if leaked
    if "<|assistant|>" in text:
        text = text.split("<|assistant|>")[-1]

    # 🔥 Capture ALL Reasoning + Conclusion blocks
    matches = re.findall(
        r"Reasoning:\s*(.*?)\s*Conclusion:\s*(.*?)(?=Reasoning:|$)",
        text,
        re.S
    )

    if not matches:
        return (
            "Reasoning:\nUnable to generate reasoning.\n\n"
            "Conclusion:\nUnable to conclude."
        )

    # ✅ TAKE ONLY THE LAST (REAL) ONE
    reasoning, conclusion = matches[-1]

    return (
        "Reasoning:\n"
        + reasoning.strip()
        + "\n\nConclusion:\n"
        + conclusion.strip()
    )
def run_inference(image_path, question):
    img = Image.open(image_path).convert("RGB")
    preds, present, absent = extract_findings_pil(img)
    prompt = build_reasoning_prompt(present, absent, question)
    answer = reason_with_phi(prompt)
    ...
    return {"preds": preds, "present": present, "absent": absent, "answer": answer}

In [10]:
result = run_inference(
    "/content/drive/MyDrive/DS3 (2)/Copy of 1612_IM-0397-1001.dcm.png",
    "Why does patient have breathing difficulty?"
)
print(result)

{'preds': {'lung_opacity': 0.3680514991283417, 'consolidation': 0.5881882905960083, 'pleural_effusion': 0.8128832578659058, 'cardiomegaly': 0.1481243520975113, 'atelectasis': 0.13433058559894562, 'edema': 0.16179592907428741, 'support_devices': 0.13569124042987823}, 'present': ['consolidation', 'pleural effusion'], 'absent': ['lung opacity', 'cardiomegaly', 'atelectasis', 'edema', 'support devices'], 'answer': "Reasoning:\nThe presence of consolidation in the lungs indicates an area of lung tissue filled with liquid instead of air, which can be due to pneumonia or other infections. This can lead to difficulty in breathing as the lungs are not able to properly exchange oxygen and carbon dioxide. The presence of pleural effusion, which is an abnormal accumulation of fluid in the pleural space, can also cause breathing difficulty. This fluid can compress the lung and limit its expansion, further impairing the patient's ability to breathe.\n\nConclusion:\nThe patient's breathing difficulty

In [6]:
import gradio as gr

def predict(image, question):

    # save uploaded image temporarily
    temp_path = "/content/temp.png"
    image.save(temp_path)

    result = run_inference(temp_path, question)

    return result


demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Image(type="pil"),
        gr.Textbox()
    ],
    outputs="json"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://538d5678e4a248946a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
